In [ ]:
import math
import requests
import chess
import chess.pgn
import chess.engine
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import Counter, defaultdict
import io
import time
import sys
import asyncio
from IPython.display import display, HTML


# Set plotting style


print("✅ All libraries imported successfully!")
print(f"🐍 Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"💻 Platform: {sys.platform}")

🪟 Windows event loop policy configured for Stockfish compatibility
✅ All libraries imported successfully!
🐍 Python 3.10.19
💻 Platform: win32


# Chess.com Game Analysis - Improve Your Game 🚀


This notebook analyzes your chess.com games to identify:
- Your most common mistakes
- Opponent weaknesses you can exploit
- Specific areas to practice

**Target:** 250 ELO rapid games

**Goal:** Get actionable insights to improve faster

# 0. Environment Setup

## 0.1. Python Version Requirements


**Recommended:** Python 3.10

**Compatibility:**
- ✅ Python 3.8 (Best compatibility)
- ✅ Python 3.9 (Recommended)
- ✅ Python 3.10 (Recommended)
- ✅ Python 3.11 (Works well)
- ⚠️ Python 3.12+ (May have issues with python-chess engine support)
- ❌ Python 3.7 or below (Not recommended - outdated)

**Why 3.9-3.11?**
- All libraries (python-chess, pandas, matplotlib) fully supported
- Stockfish engine integration works smoothly
- Good performance and stability
- Easy installation of dependencies


# 1. Data Collection

Enter your chess.com username below. The API is public and doesn't require authentication.

**API Documentation:** https://www.chess.com/news/view/published-data-api

## 1.1. Setup, Globals & Imports

In [1]:
STOCKFISH_PATH = r"D:\\softs\\stockfish\\stockfish-windows-x86-64-avx2.exe"
USERNAME = "wekies"  # Replace with your chess.com username
BASE_API_URL = "https://api.chess.com/pub"

In [2]:
from chess_tools import parse, evaluate, stockfish_setup
from analysis_tools import setup, fetch_games

In [3]:
setup.basic()
stockfish_setup.basic_setup()

%load_ext autoreload
%autoreload 2

## 1.2. Fetch Games

In [4]:
raw_games = fetch_games.filtered(
    USERNAME,
    BASE_API_URL,
    num_months=5,   # lookback window
    time_control='600',  # 10 min games
    time_class='rapid'  # rapid games
)



Found monthly archives from 2025-08 to 2025-11

Total games fetched: 144
Total 600 rapid games: 122


In [5]:
games = parse.pgn_games(raw_games)

Parsed 122 games


## 1.3. Games Summary

In [6]:
games_df = parse.generate_dataframe(games)

In [7]:
from analysis_tools import performance_calcs

# Generate and display the summary
summary = performance_calcs.summarize_performance(games_df)
performance_calcs.print_performance_summary(summary)

📊 CHESS PERFORMANCE SUMMARY

                          OVERALL STATISTICS                          
----------------------------------------------------------------------
Total Games:        122
Wins:               122 (100.0%)
Losses:             0 (0.0%)
Draws:              0 (0.0%)

                          RATING STATISTICS                           
----------------------------------------------------------------------
Current Rating:     225
Highest Rating:     678
Lowest Rating:      184
Average Rating:     252.8
Rating Change:      -453 (over 122 games)

                         PERFORMANCE BY COLOR                         
----------------------------------------------------------------------
As White:           60/60 wins (100.0%)
As Black:           62/62 wins (100.0%)

                      OPPONENT STRENGTH ANALYSIS                      
----------------------------------------------------------------------
Avg Opponent:       247
Strongest Opponent: 668
Weakest Opponent:

# 2. Game Evaluation

In [8]:
sample_game = games[0]

In [9]:
pgn_string = sample_game.metadata['pgn']

In [10]:
import chess.pgn
from io import StringIO

pgn_io = StringIO(pgn_string)
game = chess.pgn.read_game(pgn_io)

evaluator = evaluate.GameEvaluator(
    engine_path=STOCKFISH_PATH,
    depth=20,  # You can adjust the depth as needed
    time_limit=None,
    multipv=5
)

In [11]:
analysis = evaluator.evaluate_game(game)

FileNotFoundError: [WinError 2] O sistema não pode encontrar o arquivo especificado